# Import Libraries

In [14]:
import os
import csv
import ast
import requests
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import LabelEncoder
pd.set_option('display.max_rows', None)

In [15]:
import pickle

# Get Packet Fields

In [16]:
def get_nested_field_names(field_element):
    nested_field_names = []
    for nested_field in field_element.findall('field'):
        field_name = nested_field.get('name').replace(".", "_")
        if field_name == '':
         #   continue
         field_name = 'nested_empty_field_name'
        nested_field_names.extend([field_name + '_name', field_name + '_showname', field_name + '_size',
                                              field_name + '_pos', field_name + '_show', field_name + '_value'])
        nested_field_names.extend(get_nested_field_names(nested_field))  # Recursively collect nested fields
    return nested_field_names

In [17]:
def get_packet_field_names(proto):
    packet_fields = []

    for field in proto.findall('field'):
        field_name = field.get('name')
        if not field_name:
            field_name = 'empty_field_name'

        field_name = field_name.replace(".", "_")

        packet_fields.extend([
            field_name + '_name',
            field_name + '_showname',
            field_name + '_size',
            field_name + '_pos',
            field_name + '_show',
            field_name + '_value'
        ])

        packet_fields.extend(get_nested_field_names(field))

    return packet_fields

# Align Packet Features

In [18]:
def align_lists(list1, list2, list2_values):
    list2_values_aligned = []
    list2_aligned = []
    for x in list1:
        if x not in list2:
            list2_values_aligned.append('')
        else:
            idx = list2.index(x)
            list2_values_aligned.append(list2_values[idx])
    return list2_values_aligned

# Make Dataframe from Packet Features

In [19]:
def get_nested_fields(field_element):
    nested_fields = []
    for nested_field in field_element.findall('field'):
        #if nested_field.get('name') == '':
        #    continue
        nested_fields.extend([
                nested_field.get('name'),
                nested_field.get('showname'),
                nested_field.get('size'),
                nested_field.get('pos'),
                nested_field.get('show'),
                nested_field.get('value')
            ])
        nested_fields.extend(get_nested_fields(nested_field))  # Recursively collect nested fields
    return nested_fields

In [20]:
def makeDataframe(xmlfile):
    dataframe_list = []
    tree = ET.parse(xmlfile)
    root = tree.getroot()
    packets = root.findall('packet')

    column_names = []
    values = []
    flag = False

    # -------- First pass: collect column names --------
    for packet in packets:
        for proto in packet.findall('proto'):
            if proto.get('name') == 'nr-rrc':
                if proto.get('hide') == 'yes':
                  continue
                print("TEST")
                print(proto.get('name'))
                columns = get_packet_field_names(proto)
                column_names = sorted(set(column_names).union(set(columns)))
                for nas_proto in proto.iter("proto"):
                    if nas_proto.get("name") == "nas-5gs":
                      print(nas_proto.get('name'))
                      columns = get_packet_field_names(nas_proto)
                      column_names = sorted(set(column_names).union(set(columns)))
    column_names = sorted(column_names)

    # -------- Second pass: extract values --------
    for packet in packets:
        for proto in packet.findall('proto'):
            if proto.get('name') == 'nr-rrc':
                if proto.get('hide') == 'yes':
                  continue
                #flag = True
                packet_fields = []
                columns = []
                print("RRC")
                # extract all fields inside NGAP
                for field in proto.findall('field'):
                  #if not field.get('name'):
                  #    continue
                  packet_fields.extend([
                      field.get('name'),
                      field.get('showname'),
                      field.get('size'),
                      field.get('pos'),
                      field.get('show'),
                      field.get('value')
                  ])
                  packet_fields.extend(get_nested_fields(field))
                columns += get_packet_field_names(proto)

                for nas_proto in proto.iter("proto"):
                  print("NAS")
                  if nas_proto.get("name") == "nas-5gs":
                    for field in nas_proto.findall("field"):
                      packet_fields.extend([
                          field.get('name'),
                          field.get('showname'),
                          field.get('size'),
                          field.get('pos'),
                          field.get('show'),
                          field.get('value')
                      ])
                      packet_fields.extend(get_nested_fields(field))
                    columns += get_packet_field_names(nas_proto)

                values.append(align_lists(column_names, columns, packet_fields))

    print("Column names:", column_names)
    print("Number of rows:", len(values))
    return list(column_names), values

# Prepare Dataframe from PCAP File

In [21]:
import subprocess

def run_tshark_to_pdml(input_file, xml_output_file):
    # Open the output file in write mode
    with open(xml_output_file, "w") as f:
        # Run tshark - read input_file, output PDML, write stdout to f
        subprocess.run(["tshark", "-r", input_file, "-T", "pdml"], stdout=f, check=True)

In [22]:
def prepare_dataframe_from_pcap_file(xml_output_file):
    #xml_output_file = input_file.replace("pcapng", "xml")
    #run_tshark_to_pdml(input_file, xml_output_file)
    try:
      column_names, values = makeDataframe(xml_output_file)
    except:
      print("error")
    df = pd.DataFrame(values, columns=column_names)
    return df

## Encode Categorical Values

In [23]:
import random

xml_folder = os.path.join(os.getcwd(), "XML_files")
xml_files = sorted([f for f in os.listdir(xml_folder) if f.lower().endswith(".xml")])

# randomize processing order (set seed for reproducibility)
random.Random(42).shuffle(xml_files)

all_frames = []
session_map = {}  # session_name -> Sequence_Number

for idx, file_name in enumerate(xml_files, start=1):
    xml_path = os.path.join(xml_folder, file_name)
    session_name = os.path.splitext(file_name)[0]   # e.g., reject_cause_27
    session_map[session_name] = idx

    try:
        df_i = prepare_dataframe_from_pcap_file(xml_path)
    except Exception as e:
        print(f"Skipping {file_name}: {e}")
        continue

    if df_i is None or df_i.empty:
        print(f"Skipping {file_name}: empty dataframe")
        continue

    # keep only sequence id per session
    df_i["Sequence_Number"] = idx
    all_frames.append(df_i)

if not all_frames:
    raise ValueError(f"No valid XML dataframes produced from folder: {xml_folder}")

# Outer join alignment across all packet feature columns
df = pd.concat(all_frames, axis=0, join="outer", ignore_index=True, sort=False)

# Keep missing as NaN for now (important)
df = df.replace(r'^\s*$', np.nan, regex=True)

# Optional: keep Sequence_Number as integer
df["Sequence_Number"] = df["Sequence_Number"].astype(int)

# Save combined CSV
out_csv = os.path.join(os.getcwd(), "csv_files", "all_sessions_aligned.csv")
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df.to_csv(out_csv, index=False)

print(f"Saved: {out_csv}")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
print("Session mapping:", session_map)

TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
RRC
NAS
NAS
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
RRC
NAS
NAS
RRC
NAS
NAS
Column names: ['e212_mcc_name', 'e212_mcc_pos', 'e212_mcc_show', 'e212_mcc_showname', 'e212_mcc_size', 'e212_mcc_value', 'e212_mnc_name', 'e212_mnc_pos', 'e212_mnc_show', 'e212_mnc_showname', 'e212_mnc_size', 'e212_mnc_value', 'empty_field_name_name', 'empty_field_name_pos', 'empty_field_name_show', 'empty_field_name_showname', 'empty_field_name_size', 'empty_field_name_value', 'gsm_a_len_name', 'gsm_a_len_pos', 'gsm_a_len_show', 'gsm_a_len_showname', 'gsm_a_len_size', 'gsm_a_len_value', 'nas-5gs_epd_name', 'nas-5gs_epd_pos', 'nas-5gs_epd_show', 'nas-5gs_epd_showname', 'nas-5gs_epd_size', 'nas-5gs_epd_value', 'nas-5gs_mm_128_5g_ea1_name', 'nas-5gs_mm_128_5g_ea1_pos', 'nas-5gs_mm_128_5g_ea1_show', 'nas-5gs_mm_128_5g_ea1_showname', 'nas-5gs_mm_128_5g_ea1_size'

## Label Dataset

In [13]:
# Rule-based labels:
# any session containing "normal_traffic" -> 0
# all other sessions -> 1
normal_seq_ids = [
    seq_id
    for session_name, seq_id in session_map.items()
    if "normal_traffic" in session_name.lower()
]

if not normal_seq_ids:
    raise ValueError("No session containing 'normal_traffic' found in session_map.")

df["label"] = (~df["Sequence_Number"].isin(normal_seq_ids)).astype(int)

# drop Session_Name if present
if "Session_Name" in df.columns:
    df = df.drop(columns=["Session_Name"])

# encode all remaining non-numeric feature columns
protected = {"Sequence_Number", "label"}
feature_cols = [c for c in df.columns if c not in protected]
df[feature_cols] = df[feature_cols].replace(r'^\s*$', np.nan, regex=True)

category_maps = {}  # col -> {raw_string_value: int_code} or None for numeric cols

for col in feature_cols:
    if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
        s = df[col].astype("string")
        uniques = [v for v in s.dropna().unique().tolist()]
        mapping = {str(v): i for i, v in enumerate(uniques)}
        df[col] = s.astype("string").map(mapping).fillna(-1).astype(int)
        category_maps[col] = mapping
    else:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        category_maps[col] = None

df[feature_cols] = df[feature_cols].fillna(-1)

ordered_cols = ["Sequence_Number", "label"] + [c for c in df.columns if c not in ["Sequence_Number", "label"]]
df = df[ordered_cols]

os.makedirs("artifacts", exist_ok=True)
with open(os.path.join("artifacts", "feature_encoding.pkl"), "wb") as f:
    pickle.dump({"feature_cols": feature_cols, "category_maps": category_maps}, f)

# quick check
print(df[["Sequence_Number", "label"]].head(20))
print(df["label"].value_counts(dropna=False))

C:\Users\Asus\AppData\Local\Temp\ipykernel_24768\1888604112.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["label"] = (~df["Sequence_Number"].isin(normal_seq_ids)).astype(int)
C:\Users\Asus\AppData\Local\Temp\ipykernel_24768\1888604112.py:27: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\Asus\AppData\Local\Temp\ipykernel_24768\1888604112.py:27: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_ca

    Sequence_Number  label
0                 1      1
1                 1      1
2                 1      1
3                 1      1
4                 1      1
5                 1      1
6                 1      1
7                 1      1
8                 1      1
9                 2      0
10                2      0
11                2      0
12                2      0
13                2      0
14                2      0
15                2      0
16                2      0
17                2      0
18                2      0
19                2      0
label
0    117
1    103
Name: count, dtype: int64


## Save processed dataset

In [24]:
df.to_csv("csv_files/all.csv", index=False)

In [14]:
import pickle

def prepare_xml_for_inference_csv(
    xml_file: str,
    artifacts_dir: str = "artifacts",
    output_csv: str = "csv_files/infer_input.csv",
    sequence_number: int = 1
):
    df_inf = prepare_dataframe_from_pcap_file(xml_file)
    if df_inf is None or df_inf.empty:
        raise ValueError(f"No rows extracted from XML: {xml_file}")

    df_inf["Sequence_Number"] = int(sequence_number)
    df_inf = df_inf.replace(r'^\s*$', np.nan, regex=True)

    with open(os.path.join(artifacts_dir, "feature_encoding.pkl"), "rb") as f:
        enc = pickle.load(f)

    feature_cols = enc["feature_cols"]
    category_maps = enc["category_maps"]

    aligned = df_inf.reindex(columns=feature_cols)

    for col in feature_cols:
        cmap = category_maps.get(col, None)
        if cmap is not None:
            aligned[col] = aligned[col].astype("string").map(cmap).fillna(-1).astype(np.float32)
        else:
            aligned[col] = pd.to_numeric(aligned[col], errors="coerce").fillna(-1).astype(np.float32)

    out_df = pd.concat([df_inf[["Sequence_Number"]], aligned], axis=1)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    out_df.to_csv(output_csv, index=False)

    missing = [c for c in feature_cols if c not in df_inf.columns]
    print(f"Saved inference CSV: {output_csv}")
    print(f"Rows: {len(out_df)}, Features used: {len(feature_cols)}")
    if missing:
        print(f"Missing trained features filled with -1: {len(missing)}")

# Example:
# prepare_xml_for_inference_csv(
#     xml_file=r"XML_files\new_capture.xml",
#     artifacts_dir="artifacts",
#     output_csv=r"csv_files\new_capture_infer.csv",
#     sequence_number=1
# )

In [10]:
def prepare_dataframe_from_pcap_file(xml_output_file):
    # Do not swallow parse errors
    column_names, values = makeDataframe(xml_output_file)
    return pd.DataFrame(values, columns=column_names)


def _encode_features_locally(
    df: pd.DataFrame,
    protected_cols: tuple[str, ...] = ("Sequence_Number",),
    force_categorical_all: bool = True
):
    """
    Independent encoding without training-context artifacts.
    - If force_categorical_all=True: every non-protected feature is encoded as category code.
    - Unknown/missing -> -1 (within this file there is no unknown concept, but NaN -> -1).
    """
    out = df.copy()
    category_maps = {}

    feat_cols = [c for c in out.columns if c not in protected_cols]
    out[feat_cols] = out[feat_cols].replace(r'^\s*$', np.nan, regex=True)

    for col in feat_cols:
        if force_categorical_all:
            s = out[col].astype("string")
            uniques = sorted([str(v) for v in s.dropna().unique().tolist()])
            mapping = {v: i for i, v in enumerate(uniques)}
            out[col] = s.map(mapping).fillna(-1).astype(np.float32)
            category_maps[col] = mapping
        else:
            if pd.api.types.is_object_dtype(out[col]) or pd.api.types.is_categorical_dtype(out[col]):
                s = out[col].astype("string")
                uniques = sorted([str(v) for v in s.dropna().unique().tolist()])
                mapping = {v: i for i, v in enumerate(uniques)}
                out[col] = s.map(mapping).fillna(-1).astype(np.float32)
                category_maps[col] = mapping
            else:
                out[col] = pd.to_numeric(out[col], errors="coerce").fillna(-1).astype(np.float32)
                category_maps[col] = None

    return out, category_maps


def prepare_xml_for_inference_csv(
    xml_file: str,
    artifacts_dir: str = "artifacts",
    output_csv: str = "csv_files/infer_input.csv",
    sequence_number: int = 1,
    encoding_mode: str = "local"  # "local" or "trained"
):
    df_inf = prepare_dataframe_from_pcap_file(xml_file)
    if df_inf is None or df_inf.empty:
        raise ValueError(f"No rows extracted from XML: {xml_file}")

    df_inf["Sequence_Number"] = int(sequence_number)
    df_inf = df_inf.loc[:, ~df_inf.columns.str.match(r"(?i)^unnamed")]

    if encoding_mode == "trained":
        with open(os.path.join(artifacts_dir, "feature_encoding.pkl"), "rb") as f:
            enc = pickle.load(f)
        feature_cols = enc["feature_cols"]
        category_maps = enc["category_maps"]

        aligned = df_inf.reindex(columns=feature_cols)
        for col in feature_cols:
            cmap = category_maps.get(col, None)
            if cmap is not None:
                aligned[col] = aligned[col].astype("string").map(cmap).fillna(-1).astype(np.float32)
            else:
                aligned[col] = pd.to_numeric(aligned[col], errors="coerce").fillna(-1).astype(np.float32)

        out_df = pd.concat([df_inf[["Sequence_Number"]], aligned], axis=1)

    elif encoding_mode == "local":
        # Independent encoding, no dataset context needed
        encoded, local_maps = _encode_features_locally(
            df_inf, protected_cols=("Sequence_Number",), force_categorical_all=True
        )
        out_df = encoded
        # optional: save local maps for reproducibility
        os.makedirs(artifacts_dir, exist_ok=True)
        with open(os.path.join(artifacts_dir, "feature_encoding_infer_local.pkl"), "wb") as f:
            pickle.dump({"feature_cols": [c for c in out_df.columns if c != "Sequence_Number"],
                         "category_maps": local_maps}, f)
    else:
        raise ValueError("encoding_mode must be 'local' or 'trained'")

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    out_df.to_csv(output_csv, index=False)
    print(f"Saved inference CSV: {output_csv}, rows={len(out_df)}, cols={len(out_df.columns)}")

In [11]:
prepare_xml_for_inference_csv(
     xml_file=r"XML_files\reject_cause_7.xml",
     encoding_mode="local")

TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
RRC
NAS
NAS
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
Column names: ['e212_mcc_name', 'e212_mcc_pos', 'e212_mcc_show', 'e212_mcc_showname', 'e212_mcc_size', 'e212_mcc_value', 'e212_mnc_name', 'e212_mnc_pos', 'e212_mnc_show', 'e212_mnc_showname', 'e212_mnc_size', 'e212_mnc_value', 'empty_field_name_name', 'empty_field_name_pos', 'empty_field_name_show', 'empty_field_name_showname', 'empty_field_name_size', 'empty_field_name_value', 'gsm_a_len_name', 'gsm_a_len_pos', 'gsm_a_len_show', 'gsm_a_len_showname', 'gsm_a_len_size', 'gsm_a_len_value', 'nas-5gs_epd_name', 'nas-5gs_epd_pos', 'nas-5gs_epd_show', 'nas-5gs_epd_showname', 'nas-5gs_epd_size', 'nas-5gs_epd_value', 'nas-5g

C:\Users\Asus\AppData\Local\Temp\ipykernel_27336\131051601.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[feat_cols] = out[feat_cols].replace(r'^\s*$', np.nan, regex=True)


Saved inference CSV: csv_files/infer_input.csv, rows=15, cols=721
